# LangChain: Agents

## Outline:

* Using built in LangChain tools: DuckDuckGo search and Wikipedia
* Defining your own tools

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Use a current model name to avoid migration warnings in modern LangChain.
llm_model = "gpt-4o-mini"

## Built-in LangChain tools

In [3]:
from contextlib import redirect_stdout
import io
import wikipedia

from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.globals import set_debug
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


In [4]:
llm = ChatOpenAI(temperature=0, model=llm_model)

agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Use the available tools when they help you answer accurately.",
        ),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)


def build_agent(tools):
    agent_runnable = create_openai_tools_agent(llm, tools, agent_prompt)
    return AgentExecutor(
        agent=agent_runnable,
        tools=tools,
        verbose=True,
        handle_parsing_errors=True,
    )

In [5]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic Python math expression. Example input: 0.25 * 300."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


@tool
def wikipedia_search(query: str) -> str:
    """Look up a topic on Wikipedia and return a short summary."""
    try:
        return wikipedia.summary(query, sentences=2, auto_suggest=False)
    except Exception:
        matches = wikipedia.search(query, results=5)
        if matches:
            return "Possible Wikipedia matches: " + ", ".join(matches)
        return "No Wikipedia results found."


tools = [calculator, wikipedia_search]

In [6]:
agent = build_agent(tools)

In [7]:
agent.invoke({"input": "What is 25% of 300?"})



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '0.25 * 300'}`


75.025% of 300 is 75.

> Finished chain.


{'input': 'What is 25% of 300?', 'output': '25% of 300 is 75.'}

## Wikipedia example

In [8]:
question = "Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). What book did he write?"
result = agent.invoke({"input": question})



> Entering new AgentExecutor chain...

Invoking: `wikipedia_search` with `{'query': 'Tom M. Mitchell'}`


Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning.Tom M. Mitchell is the author of the textbook "Machine Learning."

> Finished chain.


## Python Agent

In [9]:
@tool
def python_repl(code: str) -> str:
    """Execute Python code for simple data processing. Use print(...) to show the final answer."""
    try:
        scope = globals().copy()
        scope["__builtins__"] = __builtins__
        buffer = io.StringIO()
        with redirect_stdout(buffer):
            exec(code, scope, scope)
        output = buffer.getvalue().strip()
        return output if output else "Code executed successfully with no printed output."
    except Exception as e:
        return f"Error: {e}"


agent = build_agent([python_repl])

In [10]:
customer_list = [["Harrison", "Chase"], 
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"], 
                 ["Geoff","Fusion"], 
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]

In [11]:
agent.invoke(
    {
        "input": f"""Sort these customers by last name and then first name.
Use Python and print the final sorted list: {customer_list}"""
    }
)



> Entering new AgentExecutor chain...

Invoking: `python_repl` with `{'code': "customers = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n\n# Sort by last name (index 0) and then by first name (index 1)\nsorted_customers = sorted(customers, key=lambda x: (x[0], x[1]))\n\nprint(sorted_customers)"}`


[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]The sorted list of customers by last name and then first name is:

```
[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]
```

> Finished chain.


{'input': "Sort these customers by last name and then first name.\nUse Python and print the final sorted list: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
 'output': "The sorted list of customers by last name and then first name is:\n\n```\n[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]\n```"}

#### View detailed outputs of the chains

In [12]:
set_debug(True)
agent.invoke(
    {
        "input": f"""Sort these customers by last name and then first name.
Use Python and print the final sorted list: {customer_list}"""
    }
)
set_debug(False)

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name.\nUse Python and print the final sorted list: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]"
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad> > chain:RunnableLambda] Entering Chain run with input:
{
  "input": ""
}
[chai

## Define your own tool

In [ ]:
# The standard library datetime module is used below; no extra install is needed.

In [13]:
from datetime import date

In [14]:
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [15]:
agent = build_agent(tools + [time])

**Note**: 

The agent will sometimes come to the wrong conclusion (agents are a work in progress!). 

If it does, please try running it again.

In [16]:
try:
    result = agent.invoke({"input": "whats the date today?"})
except Exception:
    print("exception on external access")



> Entering new AgentExecutor chain...

Invoking: `time` with `{'text': ''}`


2026-03-16Today's date is March 16, 2026.

> Finished chain.


Reminder: Download your notebook to you local computer to save your work.